In [4]:
import numpy as np
import pyvisa
import datetime
import matplotlib.pyplot as plt
import os
import pandas as pd
import time

In [ ]:
### Magnet Connection
#import for AMI PSUs
import qcodes as qc
#VISA AMI drivers
from qcodes.instrument_drivers.american_magnetics.AMI430_visa import AMI430, AMI430_3D


## Magnet functions

In [2]:
#Magnet functions

def AMI_data_collect(instr):
    #Collect data into a dictionary and returns in data frame
    data = {
        "State": str(instr),
        "Current Limit": instr.current_limit(),
        "Current Ramp Limit": instr.current_ramp_limit(),
        "Coil Constant": instr.coil_constant(),
        "Current": instr.get_cs(),
        "Calculated Supply Current": instr.field() / instr.coil_constant(),
        "Supply Voltage": instr.get_vs(),
        "Calculated System Resistance": instr.get_vs() / (instr.field() /instr.coil_constant()),
        "Calculated Field": instr.field()
        }
    data_df = pd.DataFrame([data])
    return data_df

In [3]:
def AMIcheck_up(instr):
    print(f"State of {str(instr)}:")
    print("Current limit: ", instr.current_limit())
    print("Current Ramp limit: ", instr.current_ramp_limit())
    print("Coil Constant: ", instr.coil_constant())
    print("\n")
    print("Current: ", instr.get_cs())
    print("Calculated Supply current: ", instr.field()/instr.coil_constant())
    print("Supply Voltage: ", instr.get_vs())
    print("Calculated System Resistance: ", instr.get_vs()/(instr.field()/instr.coil_constant()))
    print("Calculated Field:", instr.field())
    return

### IP address of both z and y axes

In [ ]:
# from qcodes.instrument_drivers.american_magnetics import AMIModel430, AMIModel4303D
iz = AMI430("z", address = "TCPIP0::10.196.50.17::7180::SOCKET") #connect to z axis
iy = AMI430("y", address = "TCPIP0::10.196.50.18::7180::SOCKET") #connect to y axis
i3d = AMI430_3D('AMI430_3D',  iy, iz,)


### current along z axes

In [ ]:
iz.add_parameter("get_vs", 
                 get_cmd ='VOLT:SUPP?', 
                 get_parser=float
)
iz.add_parameter("get_cs", 
                 get_cmd ='CURR:SUPP?', 
                 get_parser=float)
print(iz.get_cs())

### current along y axes

In [ ]:
iy.add_parameter("get_vs", 
                 get_cmd ='VOLT:SUPP?', 
                 get_parser=float
)
iy.add_parameter("get_cs", 
                 get_cmd ='CURR:SUPP?', 
                 get_parser=float)
print(iy.get_cs())

In [ ]:

# Set current limit
limit = 26 # Keep it small for test
iz.current_limit(limit)   
iz.current_ramp_limit(0.5) 
# Set current constant
# From Measurement- note it would be worth doing another check at some point.
cc = 1 ##  0.0235X2
iz.coil_constant(cc) 

In [ ]:
# Set current limit
limit = 26 # Keep it small for test
iy.current_limit(limit)   
iy.current_ramp_limit(0.5) 
# Set current constant
# From Measurement- note it would be worth doing another check at some point.
cc = 1 ##  0.0235X2
iy.coil_constant(cc) 

In [ ]:
## Check Magnet
AMIcheck_up(iy)
AMIcheck_up(iz)

In [ ]:
iy.close()
#iz.close()

# Fieldfox data

In [5]:

FQDN = "A-n9918A-05102.research.sydney.edu.au"
IP = '10.196.50.23'
resource_string = f"TCPIP::{IP}::inst0::INSTR"
rm = pyvisa.ResourceManager()
Trace1 = 'S21'
Trace2 = 'S21'
Trace1_format = 'MLOG'
Trace2_format = 'UPHAS'
Num_Points = 4001
IF_BW = 1000
try:
    ff = rm.open_resource(resource_string)
    ff.timeout = 5000  # in milliseconds
# Identifying mode
    idn = ff.query("*IDN?").strip()
    mode = ff.query("INST:SEL?").strip()
    print("Connected to FieldFox.")
    print(f"Instrument ID   : {idn}")
    print(f"Instrument Mode : {mode}")
except Exception as e:
    print("Error communicating with FieldFox:")
    print(e)

Connected to FieldFox.
Instrument ID   : Keysight Technologies,N9918A,MY53105102,A.08.01
Instrument Mode : "NA"


In [ ]:
folder = "C:/Users/Luigi/Documents/Gargi/13042026_Measurement_YSO/field_rotation"
os.makedirs(folder, exist_ok=True)

def save_data(Freqs, Mag_par2, Mag_par1, power_dbm):

    now = datetime.datetime.now()
    nowStr = now.strftime("%Y-%m-%d_%H-%M-%S")

    filename = f"S21_{power_dbm:+.1f}dBm_{nowStr}.csv"

    Data_Array = np.column_stack((Freqs, Mag_par2, Mag_par1))

    File_header = "Freq_Hz,Phase_deg,Mag_dB"

    full_path = os.path.join(folder, filename)

    np.savetxt(full_path, Data_Array, delimiter=",", header=File_header, comments='')

    print(f"Saved: {full_path}")

    return Data_Array

### ramp field to zero

In [ ]:
## Ramp to Field - experiment
for fld in flds:
    try:
        print("##########RAMPING###########")
        print("RAMPING FIELD TO: ", fld, " T")
        iy.field(fld)
        print(f"At {fld} T. PSU State:")
        print("##########STATE check 1 ###########")
        AMIcheck_up(iy)
        print("\n")
    except Exception as error:
        print(datetime.datetime.now())
        print("An error occurred at ", fld, " T")
        print(error)
        iy.field(0)
        AMIcheck_up(iy)
        print("Magnet at 0")
#iy.field(0) # ramp to Zero


In [ ]:

def get_AMI430_field(magnet):
    
    _, _, _, i3d = magnet
    return i3d.spherical_measured()[0] * 1e3  # T -> mT


def set_AMI430_field(magnet, field_mT, theta, phi, step_mT=1.0, sleep_time=5.0, verbose=True):
    """
    Ramp the AMI430 to a target field in mT at fixed theta and phi.

    Parameters
    ----------
    magnet : tuple
        Expected as (ix, iy, iz, i3d)
    field_mT : float
        Target field in mT
    theta : float
        Polar angle
    phi : float
        Azimuthal angle
    step_mT : float, optional
        Ramp step size in mT
    sleep_time : float, optional
        Wait time after reaching target
    verbose : bool, optional
        Print progress

    Returns
    -------
    float
        Final measured field in mT
    """
    _, _, _, i3d = magnet

    start_field_T = i3d.spherical_measured()[0]
    target_field_T = field_mT / 1000.0
    step_T = abs(step_mT) / 1000.0

    if step_T == 0:
        raise ValueError("step_mT must be > 0")

    if verbose:
        print(f"Starting at field {start_field_T*1e3:.2f} mT")

    delta = target_field_T - start_field_T
    if np.isclose(delta, 0):
        if verbose:
            print("Field already at target.")
        if sleep_time:
            time.sleep(sleep_time)
        return get_AMI430_field(magnet)

    sign = np.sign(delta)
    fields_T = np.arange(start_field_T, target_field_T, sign * step_T)

    # remove starting point and force exact final point
    if len(fields_T) > 0:
        fields_T = fields_T[1:]
    fields_T = np.append(fields_T, target_field_T)

    for f_T in fields_T:
        if verbose:
            print(f"Setting field to {f_T*1e3:.2f} mT")
        i3d.spherical([f_T, theta, phi])

    if sleep_time:
        time.sleep(sleep_time)

    measured_field_mT = get_AMI430_field(magnet)

    if verbose:
        print(f"Final measured field = {measured_field_mT:.3f} mT")

    return measured_field_mT


def sweep_AMI430_fields(magnet, fields_mT, theta, phi, step_mT=1.0, sleep_time=5.0, verbose=True):
    """
    Sweep over a list of AMI430 field values.

    Returns
    -------
    list[float]
        Measured fields in mT
    """
    measured_fields = []

    for field_mT in fields_mT:
        if verbose:
            print(f"\nTarget field: {field_mT:.2f} mT")
        measured = set_AMI430_field(
            magnet=magnet,
            field_mT=field_mT,
            theta=theta,
            phi=phi,
            step_mT=step_mT,
            sleep_time=sleep_time,
            verbose=verbose,
        )
        measured_fields.append(measured)

    return measured_fields

In [ ]:
magnet = (iy, iz, i3d)
get_AMI430_field(magnet, magnet_type="AM1430")

In [ ]:
theta = 0
phi = 90
magnet_options = {"phi": phi, "theta": theta, "step":0.5}
_ =  set_AMI430_field(magnet, 0, magnet_type="AM1430", sleep_time=5, options=magnet_options)

In [9]:
def stepTowards(targetB,theta,phi,step=5E-3):
    start_field = i3d.spherical()[0]
    end_field = targetB
    sgn=np.sign(end_field-start_field)
    #print(sgn)
    fields = np.arange(start_field,end_field+sgn*step/2,sgn*step)
    #print(fields)
    for field in fields:
        print("Setting field to %s"%(np.round(field,6)))
        i3d.spherical([field, theta, phi])

In [ ]:
def fieldvec(base_field, r01=[10,10], theta01=[0,0], phi01=[0,0], n=12):
    r_vals_T     = np.linspace(r01[0],     r01[1],     n) / 1000
    theta_vals = np.linspace(theta01[0], theta01[1], n)
    phi_vals   = np.linspace(phi01[0],   phi01[1],   n)

    return np.column_stack((r_vals_T, theta_vals, phi_vals))


In [ ]:
n=12
current_field = [iy.field(),iz.field()]
fields = fieldvec(current_field, r01 = [20,20],theta01 = [-10,10],phi01 = [90,90], n = n) 
r,theta,phi = np.transpose(fields)